<a href="https://colab.research.google.com/github/nagaraj1696/DLNN_exps_R24EV028/blob/main/exp_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# DESIGNING MULTI-LAYER PERCEPTRONS (MLP)
# FOR CLASSIFICATION AND REGRESSION
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
import keras_tuner as kt

from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD

from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, classification_report


# ============================================================
# PART A - REGRESSION
# California Housing Dataset
# ============================================================

print("\n================ PART A: REGRESSION ================\n")

# Load California Housing dataset
housing = fetch_california_housing()

X = housing.data
y = housing.target

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

# Feature scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ------------------------------------------------------------
# Build MLP model for regression
# ------------------------------------------------------------

def build_regression_model(hp):

    model = Sequential()

    # Input layer + first hidden layer
    model.add(
        Dense(
            units=hp.Int(
                "units_1",
                min_value=32,
                max_value=128,
                step=32
            ),
            activation="relu",
            input_shape=(X_train.shape[1],)
        )
    )

    # Number of additional hidden layers
    num_layers = hp.Int(
        "num_layers",
        min_value=1,
        max_value=3
    )

    for i in range(num_layers - 1):

        model.add(
            Dense(
                units=hp.Int(
                    f"units_{i+2}",
                    min_value=32,
                    max_value=128,
                    step=32
                ),
                activation="relu"
            )
        )

        model.add(
            Dropout(
                hp.Float(
                    "dropout",
                    min_value=0.0,
                    max_value=0.5,
                    step=0.1
                )
            )
        )

    # Output layer for regression
    model.add(Dense(1, activation="linear"))

    # Optimizer selection
    optimizer_name = hp.Choice(
        "optimizer",
        ["adam", "sgd"]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        [0.001, 0.0005, 0.0001]
    )

    if optimizer_name == "adam":
        optimizer = Adam(learning_rate=learning_rate)
    else:
        optimizer = SGD(learning_rate=learning_rate)

    # Compile
    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    return model


# ------------------------------------------------------------
# Hyperparameter tuning
# ------------------------------------------------------------

regression_tuner = kt.RandomSearch(
    build_regression_model,
    objective="val_loss",
    max_trials=10,
    directory="mlp_tuning",
    project_name="regression"
)

regression_tuner.search(
    X_train,
    y_train,
    epochs=30,
    validation_split=0.2,
    verbose=1
)

# Get best model
best_regression_model = regression_tuner.get_best_models(
    num_models=1
)[0]

# Display best hyperparameters
print("\nBest Regression Hyperparameters:")

best_regression_hp = regression_tuner.get_best_hyperparameters(
    num_trials=1
)[0]

print("Number of hidden layers:",
      best_regression_hp.get("num_layers"))

print("Learning rate:",
      best_regression_hp.get("learning_rate"))

print("Optimizer:",
      best_regression_hp.get("optimizer"))

print("Dropout:",
      best_regression_hp.get("dropout"))


# ------------------------------------------------------------
# Train best regression model
# ------------------------------------------------------------

best_regression_model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


# ------------------------------------------------------------
# Regression prediction
# ------------------------------------------------------------

y_pred = best_regression_model.predict(X_test).flatten()

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nRegression Results")
print("----------------------------")
print("Mean Squared Error :", mse)
print("Mean Absolute Error:", mae)
print("R2 Score           :", r2)

Trial 10 Complete [00h 00m 40s]
val_loss: 0.4079633355140686

Best val_loss So Far: 0.29416221380233765
Total elapsed time: 00h 07m 21s

Best Regression Hyperparameters:
Number of hidden layers: 3
Learning rate: 0.0005
Optimizer: adam
Dropout: 0.1
Epoch 1/30


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


413/413 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.2908 - mae: 0.3720 - val_loss: 0.3059 - val_mae: 0.3704
Epoch 2/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2859 - mae: 0.3699 - val_loss: 0.3014 - val_mae: 0.3759
Epoch 3/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2823 - mae: 0.3675 - val_loss: 0.3113 - val_mae: 0.3713
Epoch 4/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2846 - mae: 0.3700 - val_loss: 0.2937 - val_mae: 0.3725
Epoch 5/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2826 - mae: 0.3679 - val_loss: 0.3039 - val_mae: 0.3726
Epoch 6/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2824 - mae: 0.3673 - val_loss: 0.3098 - val_mae: 0.3784
Epoch 7/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2783 - mae: 0.3650 - val_loss: 0.2952 - val_mae: 0.3654
Epoch 8/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2776 - mae: 0.3646 - val_loss: 0.2861 - val_mae: 0.3624
Epoch 9/30
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.2751 

In [ ]:
# ============================================================
# PART B - CLASSIFICATION
# Iris Dataset
# ============================================================

print("\n================ PART B: CLASSIFICATION ================\n")

# Load Iris dataset
iris = load_iris()

X = iris.data
y = iris.target

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Feature scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ------------------------------------------------------------
# Build MLP model for classification
# ------------------------------------------------------------

def build_classification_model(hp):

    model = Sequential()

    # First hidden layer
    model.add(
        Dense(
            units=hp.Int(
                "units_1",
                min_value=32,
                max_value=128,
                step=32
            ),
            activation="relu",
            input_shape=(X_train.shape[1],)
        )
    )

    # Number of hidden layers
    num_layers = hp.Int(
        "num_layers",
        min_value=1,
        max_value=3
    )

    for i in range(num_layers - 1):

        model.add(
            Dense(
                units=hp.Int(
                    f"units_{i+2}",
                    min_value=32,
                    max_value=128,
                    step=32
                ),
                activation="relu"
            )
        )

        model.add(
            Dropout(
                hp.Float(
                    "dropout",
                    min_value=0.0,
                    max_value=0.5,
                    step=0.1
                )
            )
        )

    # Output layer
    model.add(
        Dense(
            3,
            activation="softmax"
        )
    )

    # Optimizer
    optimizer_name = hp.Choice(
        "optimizer",
        ["adam", "sgd"]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        [0.001, 0.0005, 0.0001]
    )

    if optimizer_name == "adam":
        optimizer = Adam(learning_rate=learning_rate)
    else:
        optimizer = SGD(learning_rate=learning_rate)

    # Compile
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# ------------------------------------------------------------
# Hyperparameter tuning
# ------------------------------------------------------------

classification_tuner = kt.RandomSearch(
    build_classification_model,
    objective="val_accuracy",
    max_trials=10,
    directory="mlp_tuning",
    project_name="classification"
)

classification_tuner.search(
    X_train,
    y_train,
    epochs=30,
    validation_split=0.2,
    verbose=1
)


# ------------------------------------------------------------
# Best classification model
# ------------------------------------------------------------

best_classification_model = classification_tuner.get_best_models(
    num_models=1
)[0]

best_classification_hp = (
    classification_tuner.get_best_hyperparameters(
        num_trials=1
    )[0]
)

print("\nBest Classification Hyperparameters:")

print("Number of hidden layers:",
      best_classification_hp.get("num_layers"))

print("Learning rate:",
      best_classification_hp.get("learning_rate"))

print("Optimizer:",
      best_classification_hp.get("optimizer"))

print("Dropout:",
      best_classification_hp.get("dropout"))


# ------------------------------------------------------------
# Train best classification model
# ------------------------------------------------------------

best_classification_model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)


# ------------------------------------------------------------
# Classification prediction
# ------------------------------------------------------------

probabilities = best_classification_model.predict(X_test)

y_pred = np.argmax(
    probabilities,
    axis=1
)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\nClassification Results")
print("----------------------------")
print("Accuracy:", accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=iris.target_names
    )
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n================ FINAL SUMMARY ================\n")

print("Regression:")
print("MSE :", mse)
print("MAE :", mae)
print("R2  :", r2)

print("\nClassification:")
print("Accuracy :", accuracy)

Trial 10 Complete [00h 00m 05s]
val_accuracy: 0.875

Best val_accuracy So Far: 0.875
Total elapsed time: 00h 00m 59s

Best Classification Hyperparameters:
Number of hidden layers: 2
Learning rate: 0.001
Optimizer: adam
Dropout: 0.0
Epoch 1/30


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step - accuracy: 0.8750 - loss: 0.3101 - val_accuracy: 0.8750 - val_loss: 0.3481
Epoch 2/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.8750 - loss: 0.2900 - val_accuracy: 0.8750 - val_loss: 0.3281
Epoch 3/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8750 - loss: 0.2756 - val_accuracy: 0.8750 - val_loss: 0.3053
Epoch 4/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8750 - loss: 0.2598 - val_accuracy: 0.8750 - val_loss: 0.2944
Epoch 5/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8750 - loss: 0.2483 - val_accuracy: 0.8750 - val_loss: 0.2823
Epoch 6/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8750 - loss: 0.2368 - val_accuracy: 0.8750 - val_loss: 0.2685
Epoch 7/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8750 - loss: 0.2270 - val_accuracy: 0.9583 - val_loss: 0.2447
Epoch 8/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8958 - loss: 0.2167 - val_accuracy: 0.9583 - val_loss: 0.2336
Epoch 9/30
